In [67]:
from typing import List, Callable, Iterator, TypeVar
import random

In [17]:
# Q: write a function in Python to add two vectors

def vector_add(z: List[float], w: List[float]) -> float:
    return [z_i + w_i for z_i, w_i in zip(z, w)]

In [8]:
# Q: write a function in Python to compute the dot product of two vectors

def vector_dot(z: List[float], w: List[float]) -> float:
    if len(z) != len(w):
        raise ValueError("Vectors must be the same length")
    return sum(z_i * w_i for z_i, w_i in zip(z, w))

In [10]:
# Q: write a function in Python to add two vectors

def vector_add(z: List[float], w: List[float]) -> List[float]:
    if len(z) != len(w):
        raise ValueError("Vectors must be the same length")
    return [z_i + w_i for z_i, w_i in zip(z, w)]

In [6]:
# Q: write a function in Python to compute the difference quotient of a scalar function at a point

def difference_quotient(f: Callable[[float], float], x: float, h:float) -> float:
    return (f(x + h) - f(x)) / h

In [11]:
# Q: Write a function in Python to get the partial difference quotient of a scalar-valued function (f: R^n \to R)

def partial_difference_quotient(
    f: Callable[[List[float]], float],
    x: List[float],
    idx: int,
    h: float
) -> float:
    h_vec = [0] * len(x)
    h_vec[idx] = h
    return (f(vector_add(x, h_vec)) - f(x)) / h

In [16]:
# Q: Write a function in Python to estimate the gradient of a scalar-valued function using partial difference quotients

def estimate_gradient(
    f: Callable[[List[float]], float],
    x: List[float],
    h: float = 0.001
) -> List[float]:
    return [partial_difference_quotient(f, x, idx, h) for idx in range(len(x))]

### Generative Questions

In [20]:
# Q: Given the following observations from a data generating process, how can you model the process using linear regression

inputs = [(x, 20 * x + 5) for x in range(-50, 50)]

**Math:**

Define 
$$
\begin{align*}
C(\theta_1, \theta_2)
&= \left( y - \begin{bmatrix} 1 & x\end{bmatrix} \begin{bmatrix} \theta_1 \\[4pt] \theta_2 \end{bmatrix} \right)^2 \\[4pt]
&= (y - \theta_1 - x\theta_2)^2
\end{align*}
$$

Our goal is to find a local minimizer of $C$. The gradient of $C$ is given by
$$
\nabla C (\theta_1, \theta_2) = \begin{bmatrix}
-2 (y - \theta_1 - x\theta_2) \\[4pt] 
-2x (y - \theta_1 - x\theta_2)
\end{bmatrix}
$$

In [35]:
def squared_cost(x: float, y: float, theta: List[float]) -> float:
    return (y - theta[0] - x * theta[1])**2

def grad_squared_cost(x: float, y: float, theta: List[float]) -> List[float]:
    error = y - theta[0] - x * theta[1]
    return [-2 * error, -2 * x * error]

In [64]:
# gradient descent

theta = [random.uniform(-1, 1) for i in range(2)]
step_size = 0.001
n_iters = 100000
tolerance = 0.000001
n = len(inputs)

for i in range(n_iters):
    cost = sum(squared_cost(x, y, theta) for x, y in inputs)
    
    # take mean of all gradients
    grad_all = [grad_squared_cost(x, y, theta) for x, y in inputs]
    grad_mean = [1/n*sum(grad[0] for grad in grad_all), 1/n*sum(grad[1] for grad in grad_all)]

    # update step
    theta = vector_add(theta, [-1 * step_size * grad_i for grad_i in grad_mean])

    if cost < tolerance:
        break

assert 19.9 < theta[1] < 20.1, "slope should be about 20"
assert 4.9 < theta[0] < 5.1, "intercept should be about 5"

In [73]:
# minibatch gradient descent

T = TypeVar('T')

def minibatches(dataset: List[T], batch_size: int, shuffle:bool = True) -> Iterator[List[T]]:
    batch_starts = [start for start in range(0, len(dataset), batch_size)]

    if shuffle: random.shuffle(batch_starts)

    for batch_start in batch_starts:
        end = batch_start + batch_size
        yield dataset[batch_start: end]

theta = [random.uniform(-1, 1) for i in range(2)]
step_size = 0.001
n_iters = 1000
tolerance = 0.0001
batch_size = 16

for i in range(n_iters):
    cost = sum(squared_cost(x, y, theta) for x, y in inputs)
    
    # update grad for each minibatch
    for batch in minibatches(inputs, batch_size, shuffle=True):
        n = len(batch)
        grad_batch = [grad_squared_cost(x, y, theta) for x, y in batch]
        grad_mean = [1/n*sum(grad[0] for grad in grad_batch), 1/n*sum(grad[1] for grad in grad_batch)]

        # update step
        theta = vector_add(theta, [-1 * step_size * grad_i for grad_i in grad_mean])

    if cost < tolerance:
        break

assert 19.9 < theta[1] < 20.1, "slope should be about 20"
assert 4.9 < theta[0] < 5.1, "intercept should be about 5"

**Note:** Put `batch_size=1` in the above snippet, and we get stochastic gradient descent